# Обучение классификатора овощей (трек A)

Перед запуском:
1. Панель справа **Input → Add Input**: датасет `vegetable-dataset` (автор preethi866).
2. **Settings → Accelerator**: GPU T4 x2.
3. **Settings → Internet** должен быть включён (в меню пункт называется «Turn off internet»): он нужен для `git clone` и весов ImageNet.

Весь ноутбук (Run All) идёт около 20–25 минут.

In [ ]:
REPO = "https://github.com/Abylcor/vegetable-classifier.git"

%cd /kaggle/working
!rm -rf vegetable-classifier && git clone -q {REPO}
%cd vegetable-classifier
!ls
!nvidia-smi -L

import sys, torch, torchvision, PIL, numpy
print("python", sys.version.split()[0], "| torch", torch.__version__, "| torchvision", torchvision.__version__,
      "| Pillow", PIL.__version__, "| numpy", numpy.__version__)

## Проверка разбиения на Kaggle (воспроизводимость)
Строим разбиение заново из файлов датасета и сравниваем с `splits.csv` из репозитория. Ожидается `совпадает: 1.0`.

Нужен scikit-learn 1.8 или новее: в более старых версиях (на Kaggle стоит такая) `StratifiedGroupKFold` с `shuffle=True` перемешивает группы с ошибкой и даёт другое, плохо стратифицированное разбиение. Поэтому сначала ставим новую версию.

In [ ]:
!pip install -q "scikit-learn>=1.8"
!python make_splits.py --data /kaggle/input --out splits_kaggle.csv
import pandas as pd
a, b = pd.read_csv("splits.csv"), pd.read_csv("splits_kaggle.csv")
key = lambda d: d["label"] + "/" + d["path"].str.split("/").str[-1]
m = a.assign(k=key(a)).merge(b.assign(k=key(b)), on="k")
print("строк:", len(a), len(b), "| сопоставлено:", len(m), "| совпадает:", (m.fold_x == m.fold_y).mean())

## Наше разбиение: 2 фолда по видео

In [ ]:
!python train.py --data /kaggle/input --splits splits.csv --test-fold 0 --out runs/fold0

In [ ]:
!python train.py --data /kaggle/input --splits splits.csv --test-fold 1 --out runs/fold1

## Итог по обоим фолдам (out-of-fold)
Каждое из 3879 фото предсказано моделью, которая не видела его видео.

In [ ]:
import sys, pandas as pd
sys.path.insert(0, ".")
from train import compute_metrics, print_metrics
from common import CLASSES
oof = pd.concat([pd.read_csv("runs/fold0/preds.csv"), pd.read_csv("runs/fold1/preds.csv")], ignore_index=True)
idx = {c: i for i, c in enumerate(CLASSES)}
print("фото:", len(oof), "| уникальных:", oof.path.nunique())
print_metrics(compute_metrics(oof.label.map(idx).tolist(), oof.pred.map(idx).tolist(), len(CLASSES)))
oof.to_csv("runs/oof_preds.csv", index=False)

## Для сравнения: авторское разбиение final_dataset
Учим на авторском train, проверяем на авторском test. Эта цифра нужна только для отчёта: она показывает, насколько утечка завышает метрику.

In [ ]:
!python train.py --data /kaggle/input --splits author_split.csv --train-folds train --test-fold test --out runs/author

## Финальная модель для predict.py
Тот же рецепт, но обучение на всех 3879 фото (тестового фолда нет: качество мы уже оценили выше).

In [ ]:
!python train.py --data /kaggle/input --splits splits.csv --test-fold none --out runs/final
!mkdir -p weights && cp runs/final/model.pt weights/model.pt && ls -la weights

## Grad-CAM
Самые уверенные ошибки (не больше одной на видео) и по одному правильному примеру каждого класса. Для каждого фото используется модель фолда, которая не видела его видео.

In [ ]:
!python gradcam.py --data /kaggle/input --runs runs --out runs/gradcam
from IPython.display import Image, display
display(Image("runs/gradcam/gradcam_grid.png"))

## predict.py на CPU
Проверяем, что скрипт работает как в ТЗ: папка на входе, CSV на выходе, битые файлы не роняют обработку. Плюс замер времени на CPU.
Фото взяты из обучающих данных, поэтому точность здесь ничего не значит: проверяем только работу скрипта и скорость.

In [ ]:
import random, shutil
from pathlib import Path
src = sorted(p for p in Path("/kaggle/input").rglob("*.jpg") if "final_dataset" not in p.parts)
photos = [p for p in src if "_frame_" not in p.name]
frames = [p for p in src if "_frame_" in p.name]
rng = random.Random(0)
test_dir = Path("/kaggle/working/predict_test")
shutil.rmtree(test_dir, ignore_errors=True)
test_dir.mkdir()
for p in rng.sample(photos, 20) + rng.sample(frames, 20):
    shutil.copy(p, test_dir / p.name)
(test_dir / "broken_empty.jpg").write_bytes(b"")
(test_dir / "broken_text.jpg").write_text("not an image")
print(len(list(test_dir.iterdir())), "файлов: 20 фото, 20 кадров, 2 битых")

In [ ]:
!python predict.py --input /kaggle/working/predict_test --output runs/predict_test.csv --benchmark 50
p = pd.read_csv("runs/predict_test.csv")
print(len(p), "строк в CSV;", p.columns.tolist())
print(p.label.value_counts().to_string())
print(p[p.label == "unknown"].to_string(index=False))

## Скачать результаты
В панели **Output** скачайте два файла до закрытия сессии:
* `results.zip` - метрики, предсказания, Grad-CAM (без весов);
* `model.pt` - веса финальной модели (около 16 МБ), их положим в репозиторий в папку `weights/`.

И сам ноутбук с выводами: **File → Download notebook**. Его кладём в репозиторий вместо `kaggle_train.ipynb`: так в нём видны все цифры этого запуска.

In [ ]:
!cd /kaggle/working/vegetable-classifier && zip -qr /kaggle/working/results.zip runs -x "*.pt"
!cp /kaggle/working/vegetable-classifier/weights/model.pt /kaggle/working/model.pt
!ls -la /kaggle/working/results.zip /kaggle/working/model.pt